<a href="https://colab.research.google.com/github/Mukundan-T/seqADAGE/blob/master/Py/muk_transfer_learning/genomic_mapping/Mukundan_sA_ec_pg_mapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E. coli &rightarrow; S. aureus Gene Mapping

### Mukundan Thanigaivelan

#### July 9, 2026

Here we are trying to map E. coli genes to S. aureus genes using techniques such as BLAST, k-mer mapping, and AlphaFold.

## 1. Connect to GitHub

In [1]:
!git clone https://github.com/Mukundan-T/seqADAGE.git

Cloning into 'seqADAGE'...
remote: Enumerating objects: 564, done.
remote: Counting objects: 100% (241/241), done.
remote: Compressing objects: 100% (218/218), done.
remote: Total 564 (delta 138), reused 49 (delta 16), pack-reused 323 (from 1)
Receiving objects: 100% (564/564), 47.10 MiB | 25.73 MiB/s, done.
Resolving deltas: 100% (291/291), done.


In [2]:
%cd seqADAGE/Py/muk_transfer_learning/genomic_mapping

/content/seqADAGE/Py/muk_transfer_learning/genomic_mapping


## 2. Loading classes & modules

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

In [5]:
# Data Analysis
import pandas as pd
import numpy as np

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Miscellaneous
import time
import tensorflow as tf
import os

In [6]:
# check CPU and GPU available in runtime
print("Num CPUs Available: ", len(tf.config.list_physical_devices('CPU')))
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print(tf.test.is_built_with_cuda())

Num CPUs Available:  1
Num GPUs Available:  1
True


In [7]:
ec_fasta = '/content/drive/MyDrive/Comp-Bio-Projs-S26/data/muk-in-use/ec_pan_genome_reference.fa'
sa_fasta = '/content/drive/MyDrive/Comp-Bio-Projs-S26/data/muk-in-use/sa_pan_genome_reference.fa'

## 3. BLAST

### Installation

In [86]:
!apt-get -qq update
!apt-get -qq install ncbi-blast+

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [87]:
!blastn -version

blastn: 2.12.0+
 Package: blast 2.12.0, build Mar  8 2022 16:19:08


### Make BLAST database

In [88]:
!makeblastdb -in "{ec_fasta}" -dbtype nucl -out ecoli_db



Building a new DB, current time: 07/20/2026 15:53:43
New DB name:   /content/seqADAGE/Py/muk_transfer_learning/genomic_mapping/ecoli_db
New DB title:  /content/drive/MyDrive/Comp-Bio-Projs-S26/data/muk-in-use/ec_pan_genome_reference.fa
Sequence type: Nucleotide
Deleted existing Nucleotide BLAST database named /content/seqADAGE/Py/muk_transfer_learning/genomic_mapping/ecoli_db
Keep MBits: T
Maximum file size: 1000000000B
Adding sequences from FASTA; added 68546 sequences in 1.5367 seconds.




### Blast S. aureus genes against database and save all hits

In [89]:
blast_hits_file = '/content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/BLAST/ec_sa_all_blast_hits.tsv'

In [90]:
!blastn \
  -query "{sa_fasta}" \
  -db ecoli_db \
  -out "{blast_hits_file}" \
  -outfmt 6

In [91]:
columns = [
  "query", "subject", "pident", "length",
  "mismatch", "gapopen", "qstart", "qend",
  "sstart", "send", "evalue", "bitscore"
]

hits = pd.read_csv(blast_hits_file, sep = "\t", names = columns)
hits.shape

(7312, 12)

### EDA on BLAST hits

In [92]:
hits.head()

,query,subject,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore
0,12b493b74dec297122c11824ed05ba01_3,panDB562_47409,99.780,1362,3,0,1,1362,1,1362,0.0,2499.0
1,12b493b74dec297122c11824ed05ba01_5,panDB562_47408,99.647,1134,4,0,1,1134,1,1134,0.0,2073.0
2,12b493b74dec297122c11824ed05ba01_9,panDB562_47407,99.820,1113,2,0,1,1113,1,1113,0.0,2045.0
3,12b493b74dec297122c11824ed05ba01_11,panDB562_47406,99.845,1935,3,0,1,1935,1,1935,0.0,3557.0
4,12b493b74dec297122c11824ed05ba01_13,panDB562_47405,98.390,2670,43,0,1,2670,1,2670,0.0,4693.0


In [93]:
hits.describe()

,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore
count,7312.000000,7312.000000,7312.000000,7312.000000,7312.000000,7312.000000,7312.000000,7312.000000,7.312000e+03,7312.000000
mean,97.386560,548.622812,11.576586,0.509847,126.768326,674.059354,216.755607,717.489606,9.927411e-09,945.319283
std,4.403544,524.543432,28.895761,2.625259,1198.731851,1302.197932,425.747971,593.130040,4.479023e-07,940.864001
min,72.301000,28.000000,0.000000,0.000000,1.000000,28.000000,1.000000,1.000000,0.000000e+00,52.800000
25%,97.143000,209.000000,1.000000,0.000000,1.000000,227.750000,1.000000,300.000000,0.000000e+00,344.000000
50%,99.142000,357.000000,3.000000,0.000000,1.000000,402.000000,1.000000,602.000000,4.800000e-166,582.000000
75%,99.822000,759.000000,10.000000,0.000000,1.000000,858.000000,271.000000,975.000000,1.775000e-94,1308.000000
max,100.000000,7176.000000,415.000000,69.000000,31333.000000,31635.000000,7012.000000,7341.000000,3.750000e-05,13075.000000


In [94]:
# Number of unique SA genes represented
hits['query'].nunique()

4356

In [95]:
# Number of unique EC genes represented
hits['subject'].nunique()

5255

In [96]:
# See number of S. aureus genes that appear with a given frequency
hits['query'].value_counts().value_counts().head()

,count
count,
1,3621
2,276
3,99
4,81
5,78


In [97]:
# See number of E. coli genes that appear with a given frequency
hits['subject'].value_counts().value_counts().head()

,count
count,
1,4398
2,388
3,227
4,127
5,61


### Filter for optimal best hits

In [98]:
# Get all the S. aureus genes with only one hit
sa_genes_with_one_hit = hits[hits['query'].map(hits['query'].value_counts()) == 1]
sa_genes_with_one_hit.shape

(3621, 12)

In [99]:
# Get all the S. aureus genes with many hits
sa_genes_with_many_hits = hits[hits['query'].map(hits['query'].value_counts()) != 1]
sa_genes_with_many_hits.shape

(3691, 12)

In [100]:
### Code generated by ChatGPT 5.5 mini
import pulp

# Save the unique SA and EC genes to NumPy arrays
unique_ec_genes = sa_genes_with_many_hits['subject'].unique()
unique_sa_genes = sa_genes_with_many_hits['query'].unique()

# Create the optimization problem
prob = pulp.LpProblem("MaximizeUniqueSubjects", pulp.LpMaximize)

# One binary variable for each possible assignment (row)
x = {
  i: pulp.LpVariable(f"x_{i}", cat="Binary")
  for i in sa_genes_with_many_hits.index
}

# One binary variable for each subject
y = {
  s: pulp.LpVariable(f"y_{s}", cat="Binary")
  for s in unique_ec_genes
}

# ----------------------------
# Constraints
# ----------------------------

# Each query must choose exactly one subject
for q in unique_sa_genes:
  rows = sa_genes_with_many_hits.index[sa_genes_with_many_hits["query"] == q]
  prob += pulp.lpSum(x[i] for i in rows) == 1

# If a row uses a subject, that subject is "used"
for s in unique_ec_genes:
  rows = sa_genes_with_many_hits.index[sa_genes_with_many_hits["subject"] == s]
  for i in rows:
    prob += x[i] <= y[s]

# ----------------------------
# Objective
# ----------------------------

# Normalize e-values so the penalty is well behaved.
# If your evalues are already transformed (e.g. -log10),
# adjust this accordingly.

e = sa_genes_with_many_hits["evalue"].astype(float)

# Small penalty for worse evalues
eps = 1e-6

prob += (
  pulp.lpSum(y.values())
  - eps * pulp.lpSum(e[i] * x[i] for i in sa_genes_with_many_hits.index)
)

# Solve
prob.solve(pulp.PULP_CBC_CMD(msg=True))

# Extract selected rows
result = sa_genes_with_many_hits[[x[i].value() > 0.5 for i in sa_genes_with_many_hits.index]].copy()
result.shape

(735, 12)

In [101]:
print(
  f"There are {result['query'].nunique()} unique S. aureus genes and {result['subject'].nunique()} unique E. coli genes."
)

There are 735 unique S. aureus genes and 576 unique E. coli genes.


In [102]:
# Now we have the best hit for each SA gene
best_hits = pd.concat([sa_genes_with_one_hit, result], ignore_index=True)
best_hits.shape

(4356, 12)

In [103]:
# We've picked up 2,735 unique E. coli genes!
best_hits['subject'].nunique()

2735

In [104]:
# Save best hits
best_hits.to_csv(
  '/content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/BLAST/ec_sa_best_blast_hits.csv',
  index = True
)

In [105]:
# Verify I can load back in data
best_hits = pd.read_csv(
  '/content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/BLAST/ec_sa_best_blast_hits.csv',
  index_col = 0
)
best_hits.shape

(4356, 12)

## 4. $k$-mer mapping

### Installation

In [106]:
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
!bash miniconda.sh -b -p /usr/local/miniconda

PREFIX=/usr/local/miniconda
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /usr/local/miniconda


In [107]:
# Add to PATH
os.environ["PATH"] += ":/usr/local/miniconda/bin"

In [108]:
!conda config --add channels defaults
!conda config --add channels bioconda
!conda config --add channels conda-forge

In [109]:
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r


In [110]:
!conda install -y blat

Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: - \ done
Channels:
 - conda-forge
 - bioconda
 - defaults
Platform: linux-64
Solving environment: - \ | done

## Package Plan ##

  environment location: /usr/local/miniconda

  added / updated specs:
    - blat


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    blat-35                    |                1         1.7 MB  bioconda
    ca-certificates-2026.6.17  |       hbd8a1cb_0         126 KB  conda-forge
    conda-26.5.3               |  py314h9e666f3_2         1.4 MB  conda-forge
    libpng-1.6.58              |       h421ea60_0         310 KB  conda-forge
    openssl-3.6.3              |       h35e630c_0         3.0 MB  conda-forge
    ------------------------------------------------------------
                                           Total:         6.5 MB

The following NEW packages will be INS

In [111]:
!blat

blat - Standalone BLAT v. 35 fast sequence search command line tool
usage:
   blat database query [-ooc=11.ooc] output.psl
where:
   database and query are each either a .fa , .nib or .2bit file,
   or a list these files one file name per line.
   -ooc=11.ooc tells the program to load over-occurring 11-mers from
               and external file.  This will increase the speed
               by a factor of 40 in many cases, but is not required
   output.psl is where to put the output.
   Subranges of nib and .2bit files may specified using the syntax:
      /path/file.nib:seqid:start-end
   or
      /path/file.2bit:seqid:start-end
   or
      /path/file.nib:start-end
   With the second form, a sequence id of file:start-end will be used.
options:
   -t=type     Database type.  Type is one of:
                 dna - DNA sequence
                 prot - protein sequence
                 dnax - DNA sequence translated in six frames to protein
               The default is dna
   -q=type     

### Run BLAT and save all hits

In [112]:
# File to store k-mer best hits
kmer_hits_file = '/content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/k-mer-mapping/ec_sa_all_kmer_hits.psl'

In [113]:
!blat \
  "{ec_fasta}" \
  "{sa_fasta}" \
  "{kmer_hits_file}"

Loaded 46157027 letters in 68546 sequences
Searched 6144633 bases in 9935 sequences


In [115]:
psl_columns = [
  "matches", "misMatches", "repMatches", "nCount",
  "qNumInsert", "qBaseInsert", "tNumInsert", "tBaseInsert",
  "strand", "qName", "qSize", "qStart", "qEnd", "tName",
  "tSize", "tStart", "tEnd", "blockCount", "blockSizes",
  "qStarts", "tStarts",
]

blat_hits = pd.read_csv(
  kmer_hits_file,
  sep = "\t",
  skiprows = 5,
  names = psl_columns,
)
blat_hits.shape

(7030, 21)

### EDA on $k$-mer hits

In [119]:
blat_hits.head()

,matches,misMatches,repMatches,nCount,qNumInsert,qBaseInsert,tNumInsert,tBaseInsert,strand,qName,...,qStart,qEnd,tName,tSize,tStart,tEnd,blockCount,blockSizes,qStarts,tStarts
0,1359,3,0,0,0,0,0,0,+,12b493b74dec297122c11824ed05ba01_3,...,0,1362,panDB562_47409,1362,0,1362,1,"1362,","0,","0,"
1,1130,4,0,0,0,0,0,0,+,12b493b74dec297122c11824ed05ba01_5,...,0,1134,panDB562_47408,1134,0,1134,1,"1134,","0,","0,"
2,1111,2,0,0,0,0,0,0,+,12b493b74dec297122c11824ed05ba01_9,...,0,1113,panDB562_47407,1113,0,1113,1,"1113,","0,","0,"
3,1932,3,0,0,0,0,0,0,+,12b493b74dec297122c11824ed05ba01_11,...,0,1935,panDB562_47406,1935,0,1935,1,"1935,","0,","0,"
4,2617,42,0,0,1,7,1,1,+,12b493b74dec297122c11824ed05ba01_13,...,0,2666,panDB562_47405,2670,0,2660,2,"2650,9,","0,2657,","0,2651,"


In [120]:
blat_hits.describe()

,matches,misMatches,repMatches,nCount,qNumInsert,qBaseInsert,tNumInsert,tBaseInsert,qSize,qStart,qEnd,tSize,tStart,tEnd,blockCount
count,7030.000000,7030.000000,7030.0,7030.000000,7030.000000,7030.000000,7030.000000,7030.000000,7030.000000,7030.000000,7030.000000,7030.000000,7030.000000,7030.000000,7030.000000
mean,532.342959,6.440825,0.0,0.006117,0.082930,3.750213,0.102276,5.615932,697.577098,99.989616,642.529730,811.418777,190.939829,735.345661,1.137269
std,527.244355,11.481178,0.0,0.132142,0.447667,44.863219,0.534679,41.583894,1320.500307,795.313384,946.004964,625.160113,403.949294,578.639056,0.664372
min,30.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,70.000000,0.000000,32.000000,63.000000,0.000000,32.000000,1.000000
25%,198.000000,1.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,234.000000,0.000000,219.000000,369.000000,0.000000,315.000000,1.000000
50%,331.500000,3.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,429.000000,0.000000,390.000000,675.000000,0.000000,609.000000,1.000000
75%,744.000000,8.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,888.000000,0.000000,854.500000,1062.000000,203.250000,981.000000,1.000000
max,7199.000000,296.000000,0.0,8.000000,11.000000,2073.000000,10.000000,799.000000,31635.000000,29478.000000,31497.000000,7341.000000,6095.000000,7341.000000,15.000000


In [121]:
# Number of unique SA genes represented
blat_hits['qName'].nunique()

4246

In [122]:
# Number of unique EC genes represented
blat_hits['tName'].nunique()

5171

In [123]:
# See number of S. aureus genes that appear with a given frequency
blat_hits['qName'].value_counts().value_counts().head()

,count
count,
1,3578
2,238
4,85
3,83
5,73


In [124]:
# See number of E. coli genes that appear with a given frequency
blat_hits['tName'].value_counts().value_counts().head()

,count
count,
1,4367
2,359
3,230
4,126
5,52


### Parse BLAT Output

In [ ]:
# Define a percentage match metric
blat_hits['pident'] = 100 * (blat_hits['matches'] / (blat_hits['matches'] + blat_hits['misMatches']))

# Custom score since we don't necessarily get a 'pident' --> reward high number of matches and high percentage of matches
blat_hits['score'] = blat_hits['matches'] * (blat_hits['pident'] / 100)
blat_hits.head()

,matches,misMatches,repMatches,nCount,qNumInsert,qBaseInsert,tNumInsert,tBaseInsert,strand,qName,...,tName,tSize,tStart,tEnd,blockCount,blockSizes,qStarts,tStarts,pident,score
0,1359,3,0,0,0,0,0,0,+,12b493b74dec297122c11824ed05ba01_3,...,panDB562_47409,1362,0,1362,1,"1362,","0,","0,",99.779736,1356.006608
1,1130,4,0,0,0,0,0,0,+,12b493b74dec297122c11824ed05ba01_5,...,panDB562_47408,1134,0,1134,1,"1134,","0,","0,",99.647266,1126.014109
2,1111,2,0,0,0,0,0,0,+,12b493b74dec297122c11824ed05ba01_9,...,panDB562_47407,1113,0,1113,1,"1113,","0,","0,",99.820305,1109.003594
3,1932,3,0,0,0,0,0,0,+,12b493b74dec297122c11824ed05ba01_11,...,panDB562_47406,1935,0,1935,1,"1935,","0,","0,",99.844961,1929.004651
4,2617,42,0,0,1,7,1,1,+,12b493b74dec297122c11824ed05ba01_13,...,panDB562_47405,2670,0,2660,2,"2650,9,","0,2657,","0,2651,",98.420459,2575.663407


In [ ]:
# Get top 2 E. coli hits for every S. aureus gene
top2_blat_hits = (
    blat_hits
    .sort_values(["qName", "score"], ascending=[True, False])
    .groupby("qName")
    .head(2)
)

top2_blat_hits.shape

(4914, 23)

In [ ]:
# We've pulled about 3,200 E. coli genes
top2_blat_hits['tName'].nunique()

3199

In [ ]:
# Save best hits
top2_blat_hits.to_csv(
  '/content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/ec_sa_kmer_hits.csv',
  index = True
)

In [ ]:
# Verify I can load back in data
blat_hits_loaded = pd.read_csv(
  '/content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/ec_sa_kmer_hits.csv',
  index_col = 0
)
blat_hits_loaded.shape

(4914, 23)

## 5. Foldseek

### Convert DNA FASTA to Protein FASTA

In [ ]:
!pip install -qq biopython

In [ ]:
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord

def translate_fasta(input_fasta, output_fasta):
  proteins = []

  for record in SeqIO.parse(input_fasta, "fasta"):
    protein = record.seq.translate(to_stop=True)

    proteins.append(
      SeqRecord(
        protein,
        id = record.id,
        description = record.description
      )
    )

  SeqIO.write(proteins, output_fasta, "fasta")

In [ ]:
translate_fasta(ec_fasta, "/content/drive/MyDrive/Comp-Bio-Projs-S26/data/muk-in-use/ec_protein.faa")
translate_fasta(sa_fasta, "/content/drive/MyDrive/Comp-Bio-Projs-S26/data/muk-in-use/sa_protein.faa")

In [ ]:
!head -10 "/content/drive/MyDrive/Comp-Bio-Projs-S26/data/muk-in-use/ec_protein.faa"

>11997cc26382b2c286cd502685a104a5_3 thrL
MKRISTTITTTITITTGNGAG
>11997cc26382b2c286cd502685a104a5_5 thrA
MRVLKFGGTSVANAERFLRVADILESNARQGQVATVLSAPAKITNHLVAMIEKTISGQDA
LPNISDAERIFAELLTGLAAAQPGFPLAQLKTFVDQEFAQIKHVLHGISLLGQCPDSINA
ALICRGEKMSIAIMAGVLEARGHNVTVIDPVEKLLAVGHYLESTVDIAESTRRIAASRIP
ADHMVLMAGFTAGNEKGELVVLGRNGSDYSAAVLAACLRADCCEIWTDVDGVYTCDPRQV
PDARLLKSMSYQEAMELSYFGAKVLHPRTITPIAQFQIPCLIKNTGNPQAPGTLIGASRD
EDELPVKGISNLNNMAMFSVSGPGMKGMVGMAARVFAAMSRARISVVLITQSSSEYSISF
CVPQSDCVRAERAMQEEFYLELKEGLLEPLAVTERLAIISVVGDGMRTLRGISAKFFAAL
